In [1]:
%load_ext autoreload
%autoreload 2

In [64]:
import pickle
import torch
import sys
import numpy as np
import yaml
from pathlib import Path
import matplotlib.pyplot as plt
from pieridae.starbursts import byol

In [17]:
sys.path.append('../../scripts/')
import run_sfourl_fire

In [21]:
logger = run_sfourl_fire.setup_logging(Path('./'), 'INFO')

In [35]:
output_path = byol.Path('../../output/fire2_sfourl/')

In [22]:
def load_config(config_path: str) -> dict:
    """Load configuration from YAML file"""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Convert paths to Path objects
    config['data']['input_path'] = Path(config['data']['input_path'])
    config['data']['output_path'] = Path(config['data']['output_path'])

    return config


In [67]:
with open(output_path / 'training_data.pkl', 'rb') as f:
    training_data = pickle.load(f)

In [30]:
images, img_names, true_labels, class_names = run_sfourl_fire.load_fire2_mock_images(
    Path('../../local_data/mock_images/'),
    ['m11h_res7100','m11d_res7100','m11b_res2100'],
    10,
    logger
)
images_processed = run_sfourl_fire.preprocess_images_for_byol(images, target_channels=3, logger=logger)

2025-12-18 16:26:46,886 - fire2_analysis - INFO - Loading FIRE2 mock images from: ../../local_data/mock_images
2025-12-18 16:26:46,886 - fire2_analysis - INFO - Loading FIRE2 mock images from: ../../local_data/mock_images
2025-12-18 16:26:46,887 - fire2_analysis - INFO - Galaxy tags: ['m11h_res7100', 'm11d_res7100', 'm11b_res2100']
2025-12-18 16:26:46,887 - fire2_analysis - INFO - Galaxy tags: ['m11h_res7100', 'm11d_res7100', 'm11b_res2100']
2025-12-18 16:26:46,897 - fire2_analysis - INFO - Loading 10 images from m11h_res7100...
2025-12-18 16:26:46,897 - fire2_analysis - INFO - Loading 10 images from m11h_res7100...
2025-12-18 16:26:46,904 - fire2_analysis - INFO -   Loaded 10 images from m11h_res7100
2025-12-18 16:26:46,904 - fire2_analysis - INFO -   Loaded 10 images from m11h_res7100
2025-12-18 16:26:46,912 - fire2_analysis - INFO - Loading 10 images from m11d_res7100...
2025-12-18 16:26:46,912 - fire2_analysis - INFO - Loading 10 images from m11d_res7100...
2025-12-18 16:26:46,918 

In [116]:
config = load_config(byol.Path('../../config.yaml'))
modelmanager = byol.BYOLModelManager(config=config, output_path=output_path, n_classes=3)
modelmanager.load_trained_model(output_path / 'best_model_checkpoint.pt',)
modelmanager.learner.train()
modelmanager.classifier.train()

2025-12-18 17:18:20,144 - INFO - Using Apple Silicon GPU (MPS): mps
2025-12-18 17:18:20,145 - INFO - Adjusted training batch size for MPS: 1024 -> 128
2025-12-18 17:18:20,145 - INFO - Adjusted inference batch size for MPS: 1024 -> 128
2025-12-18 17:18:20,146 - INFO - BYOLModelManager initialized on device: mps
2025-12-18 17:18:20,146 - INFO - Loading trained model from: ../../output/fire2_sfourl/best_model_checkpoint.pt
2025-12-18 17:18:20,146 - INFO - Setting up BYOL model...
2025-12-18 17:18:20,540 - INFO - Created classification head: 512 -> 3 classes
2025-12-18 17:18:20,541 - INFO - BYOL model setup complete on mps
2025-12-18 17:18:20,743 - INFO - Loaded classifier from checkpoint
2025-12-18 17:18:20,744 - INFO - Model loaded successfully


Linear(in_features=512, out_features=3, bias=True)

In [117]:
embeddings = modelmanager.extract_embeddings(training_data['training_images'])

2025-12-18 17:18:20,863 - INFO - Extracting embeddings from 150 images...
Extracting embeddings: 100%|████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 17.83it/s]
2025-12-18 17:18:21,035 - INFO - Extracted embeddings shape: (150, 512)


In [118]:
classifier = byol.FrozenClassifier(
    model_path=output_path / 'best_model_checkpoint.pt',
    config=config,
    logger=logger,
    n_classes=3     
)


2025-12-18 17:18:22,575 - fire2_analysis - INFO - FrozenClassifier using device: mps
2025-12-18 17:18:22,575 - fire2_analysis - INFO - FrozenClassifier using device: mps
2025-12-18 17:18:22,577 - fire2_analysis - INFO - Loading trained classifier from: ../../output/fire2_sfourl/best_model_checkpoint.pt
2025-12-18 17:18:22,577 - fire2_analysis - INFO - Loading trained classifier from: ../../output/fire2_sfourl/best_model_checkpoint.pt
2025-12-18 17:18:23,135 - fire2_analysis - INFO - Loaded classifier: 512 -> 3 classes
2025-12-18 17:18:23,135 - fire2_analysis - INFO - Loaded classifier: 512 -> 3 classes


In [137]:
modelmanager.learner.eval()
modelmanager.classifier.eval()

# Extract embeddings with SAME chunk size as training (16)
chunk_size = 16  # Must match supervised_chunk_size from training!
all_embeddings = []
images = training_data['training_images']

with torch.no_grad():
    for i in range(0, len(images), chunk_size):
        batch = torch.tensor(
            images[i:i+chunk_size],
            dtype=torch.float32
        ).to(modelmanager.device)
        
        _, emb = modelmanager.learner(batch, return_embedding=True)
        all_embeddings.append(emb.cpu().numpy())

embeddings = np.vstack(all_embeddings)

# Now compute loss
batch = torch.tensor(embeddings, dtype=torch.float32).to(modelmanager.device)
logits = modelmanager.classifier(batch)
loss = torch.nn.functional.cross_entropy(
    logits, 
    torch.tensor(
        training_data['training_labels'][training_data['training_labels']>0]-1, 
        dtype=torch.long
    ).to(modelmanager.device)
)
print(f"Loss with chunk_size=16: {loss.item()}")

Loss with chunk_size=16: 0.8274357318878174


In [135]:
iterative_labels, n_labels_iter, prob_labels_iter, stats = \
    classifier.iterative_propagation(embeddings, true_labels)

AttributeError: 'FrozenClassifier' object has no attribute 'train'

In [131]:
batch = torch.tensor(
    embeddings,
    dtype=torch.float32
).to(modelmanager.device)

In [132]:
logits = modelmanager.classifier(batch)

In [133]:
probs = torch.softmax(logits, dim=1).cpu().detach().numpy()

In [134]:
torch.nn.functional.cross_entropy(
    logits, 
    torch.tensor(
        training_data['training_labels'][training_data['training_labels']>0]-1, 
        dtype=torch.long
    ).to(modelmanager.device)
)

tensor(0.3255, device='mps:0', grad_fn=<NllLossBackward0>)